# Multiplier-free sequence modeling, and recovering accuracy for free

The last place a multiply hides in a "closed" model is the **loss**: a softmax
cross-entropy uses exp + divide + multiply. On silicon the loop instead used a
*perceptron* gradient (a 2-hot ±1 vector) — multiplier-free, but the single
biggest accuracy cost in the closure penalty (~−17pp on char-LM).

TikiTorch's `tropical_cross_entropy` closes that gap **without a hardware
multiply**: the CE gradient is `softmax(logits) − onehot(target)`, and
`shift_softmax` computes the softmax from a subtract-max, a power-of-2-addressed
exp lookup, additions, and a shift-normalize. This notebook reproduces the
recovery live on a character LM, then shows the `TropicalLion` optimizer and the
streaming `MorphSSM` cell.


In [ ]:
# --- Setup: install TikiTorch (run once) ---
# TikiTorch is research code, not on PyPI. In Colab, install from the repo.
# Replace REPO_URL with your TikiTorch checkout (it contains tikitorch/ + data/).
try:
    import tikitorch  # already installed?
except ImportError:
    REPO_URL = "https://github.com/your-org/tikitorch.git"  # <-- set me
    import subprocess, sys, os
    if not os.path.exists("tikitorch_repo"):
        subprocess.run(["git", "clone", REPO_URL, "tikitorch_repo"], check=False)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "tikitorch_repo/tikitorch"], check=False)
    import tikitorch
import torch
print("TikiTorch", getattr(tikitorch, "__version__", "?"), "| torch", torch.__version__)


## A character-level Shakespeare LM at the silicon dims (D=128, V=128)


In [ ]:
import torch, tikitorch as tt, tikitorch.optim as optim
V = 128

# Tiny built-in text so the notebook is self-contained; swap in any corpus.
import urllib.request, os
url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
try:
    text = urllib.request.urlopen(url, timeout=10).read().decode("utf-8", "replace")
except Exception:
    text = ("To be, or not to be, that is the question:\n" * 2000)   # fallback
ids = torch.tensor([min(ord(c), V - 1) for c in text], dtype=torch.long)
print("corpus chars:", ids.numel())

def batch(bs, L, gen):
    s = torch.randint(0, ids.numel() - L - 1, (bs,), generator=gen)
    x = torch.stack([ids[i:i+L] for i in s]); y = torch.stack([ids[i+1:i+L+1] for i in s])
    return x, y


## The recovery: perceptron → closure-faithful CE (multiplier-free)

We hold the model, optimizer, seed, and data fixed and vary **only the loss
conditioning**: the perceptron baseline (what silicon used), the host-softmax
`tropical_ce`, and the fully multiplier-free `tropical_ce_lut`. Watch the
multiplier-free LUT softmax recover most of the perceptron penalty — and match
the float cross-entropy ceiling (under a sign optimizer, only the gradient *sign*
matters, and the LUT reproduces it).


In [ ]:
def run(loss_type, steps=400, L=48, bs=24, seed=0):
    torch.manual_seed(seed); gen = torch.Generator().manual_seed(seed)
    m = tt.models.TropicalTransformer(d_model=128, n_heads=4, n_layers=1, d_ff=512,
                                      tau=0.5, max_seq_len=L, task_type="lm", vocab_size=V)
    m.loss_type = loss_type
    opt = optim.SignSGD(m.parameters(), lr=2e-3, momentum=0.9)
    for _ in range(steps):
        x, y = batch(bs, L, gen); opt.zero_grad()
        m.compute_loss(x, y).backward(); opt.step()
    gen2 = torch.Generator().manual_seed(seed + 1)
    accs = []
    with torch.no_grad():
        for _ in range(8):
            x, y = batch(bs, L, gen2)
            accs.append((m(x).reshape(-1, V).argmax(-1) == y.reshape(-1)).float().mean().item())
    return sum(accs) / len(accs)

rows = [("perceptron (silicon baseline)", "perceptron"),
        ("tropical_ce (host softmax)", "tropical_ce"),
        ("tropical_ce_lut (FULLY mult-free)", "tropical_ce_lut"),
        ("cross_entropy (float ceiling)", "cross_entropy")]
res = {lt: run(lt) for _, lt in rows}
for label, lt in rows:
    print(f"{label:34s}  eval_acc={res[lt]:.4f}")
print(f"\nrecovery perceptron -> tropical_ce_lut: "
      f"{res['tropical_ce_lut'] - res['perceptron']:+.4f}  (multiplier-free)")


## A multiplier-free momentum optimizer: TropicalLion

`TropicalLion` takes the **sign of a momentum** (stronger than SignSGD's
sign-of-gradient), with power-of-2 betas so the EMA is a shift+add. Crucially the
momentum is simulated in **truncating integer fixed-point**, faithfully modeling
the silicon (a float EMA would over-state it). `momentum_density()` confirms the
momentum is alive (not collapsed to SignSGD).


In [ ]:
m = tt.models.TropicalTransformer(d_model=128, n_heads=4, n_layers=1, d_ff=512,
                                  tau=0.5, max_seq_len=48, task_type="lm", vocab_size=V)
m.loss_type = "tropical_ce"
lion = optim.TropicalLion(m.parameters(), lr_shift=9, grad_bits=16)
gen = torch.Generator().manual_seed(0)
for _ in range(50):
    x, y = batch(24, 48, gen); lion.zero_grad(); m.compute_loss(x, y).backward(); lion.step()
print("Lion momentum density (fraction of live coords):", round(lion.momentum_density(), 3),
      "\n> 0 means a real momentum, not SignSGD-degenerate.")


## Streaming inference: MorphSSM `.step()`

The morphological SSM runs token-by-token with an on-chip ring buffer — the form
the FPGA kernel uses for streaming. Same weights, no recompute.


In [ ]:
ssm = tt.models.MorphSSM(d_model=64, n_layers=2, task_type="lm", vocab_size=V)
from tikitorch.audit import find_closure_violations
print("MorphSSM closure violations (LayerNorm/Linear-head are the declared boundary):",
      [t for _, t in find_closure_violations(ssm)][:4])
print("Swap in TropicalLayerNorm + a tropical head for a fully-closed sequence model.")
